**Viz 5:** Crime by distance from Snell (Altair area or line chart, interactive tooltip)

Notes: Compute each incident’s distance from Snell and bin it into 0.05-mile rings. Plot counts per ring, with separate lines for day and night. This tells students whether risk rises as they walk away from campus toward Mission Hill or Roxbury Crossing, or whether it’s concentrated right at the campus edge.

In [1]:
import altair as alt
import numpy as np
import pandas as pd

# utf-8-sig strips the hidden BOM character at the start of the file
crime = pd.read_csv('crime.csv', encoding='utf-8-sig')
crime.head()

,INCIDENT_NUMBER,OFFENSE_CODE,OFFENSE_CODE_GROUP,OFFENSE_DESCRIPTION,DISTRICT,REPORTING_AREA,SHOOTING,OCCURRED_ON_DATE,YEAR,MONTH,DAY_OF_WEEK,HOUR,UCR_PART,STREET,Lat,Long,Location
0,I182079978,619,Larceny,LARCENY ALL OTHERS,C11,817,NaN,6/4/18 9:00,2018,6,Monday,9,Part One,TALBOT AVE,42.287148,-71.064930,"(42.28714820, -71.06493039)"
1,I182079349,727,Auto Theft,AUTO THEFT - LEASED/RENTED VEHICLE,C11,404,NaN,8/12/18 22:00,2018,8,Sunday,22,Part One,VALLEY RD,42.277431,-71.068511,"(42.27743055, -71.06851083)"
2,I182079339,617,Larceny,LARCENY THEFT FROM BUILDING,D4,152,NaN,8/11/18 13:13,2018,8,Saturday,13,Part One,TREMONT ST,42.343934,-71.071688,"(42.34393403, -71.07168782)"
3,I182079317,724,Auto Theft,AUTO THEFT,A7,11,NaN,8/10/18 13:47,2018,8,Friday,13,Part One,WILLIAM F MCCLELLAN HWY,42.393529,-71.011582,"(42.39352862, -71.01158244)"
4,I182079315,724,Auto Theft,AUTO THEFT,A7,11,NaN,7/30/18 14:05,2018,7,Monday,14,Part One,WILLIAM F MCCLELLAN HWY,42.393529,-71.011582,"(42.39352862, -71.01158244)"


### 1. Clean the data

`crime.csv` is already filtered to the project's crime types, so the only cleaning here is:
- drop rows with missing or placeholder coordinates (-1 / 0)
- keep only incidents within **0.5 miles** of campus (done in step 2)
- count every offense row (not unique incident numbers)

In [2]:
df = crime.dropna(subset=['Lat', 'Long'])
df = df[(df['Lat'] > 40) & (df['Long'] < -70)]   # removes -1 / 0 placeholder coordinates

print(f'{len(crime):,} rows -> {len(df):,} rows after removing bad coordinates')

17,989 rows -> 17,179 rows after removing bad coordinates


### 2. Distance from Snell, rings, and day vs. night

Distance uses the haversine formula (straight-line distance on the Earth's surface, in miles).
Each incident is binned into a 0.05-mile ring: ring 0.10 means 0.10–0.15 mi from Snell.

**Day** = 6:00 AM – 5:59 PM, **Night** = 6:00 PM – 5:59 AM.

In [3]:
SNELL_LAT, SNELL_LON = 42.33841, -71.08809   # Snell Library
RADIUS_MI = 0.5
RING_MI = 0.05
EARTH_RADIUS_MI = 3958.8

def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * EARTH_RADIUS_MI * np.arcsin(np.sqrt(a))

df = df.copy()
df['dist_mi'] = haversine_miles(SNELL_LAT, SNELL_LON, df['Lat'], df['Long'])
near = df[df['dist_mi'] < RADIUS_MI].copy()

# ring start in miles, rounded to avoid floating point issues like 0.15000000002
near['ring_start'] = (np.floor(near['dist_mi'] / RING_MI) * RING_MI).round(2)
near['period'] = np.where(near['HOUR'].between(6, 17), 'Day', 'Night')

print(f'{len(near):,} incidents within {RADIUS_MI} mi of Snell')
near['period'].value_counts()

551 incidents within 0.5 mi of Snell


period
Day      319
Night    232
Name: count, dtype: int64

### 3. Count incidents per ring

Every ring × period combination is included, even if it has 0 incidents, so the lines don't skip rings.

Outer rings cover more ground than inner rings, so they will naturally have more incidents.
To make rings comparable, the tooltip also shows **incidents per square mile** (count ÷ ring area).

In [4]:
rings = np.round(np.arange(0, RADIUS_MI, RING_MI), 2)
grid = pd.MultiIndex.from_product([rings, ['Day', 'Night']],
                                  names=['ring_start', 'period']).to_frame(index=False)

counts = near.groupby(['ring_start', 'period']).size().rename('incidents').reset_index()

top_offense = (near.groupby(['ring_start', 'period'])['OFFENSE_CODE_GROUP']
                   .agg(lambda s: s.value_counts().idxmax())
                   .rename('top_offense').reset_index())

ring_df = (grid.merge(counts, how='left').merge(top_offense, how='left')
               .fillna({'incidents': 0, 'top_offense': 'None'}))
ring_df['incidents'] = ring_df['incidents'].astype(int)

ring_df['ring_end'] = (ring_df['ring_start'] + RING_MI).round(2)
ring_df['ring_label'] = ring_df.apply(
    lambda r: f"{r['ring_start']:.2f}–{r['ring_end']:.2f} mi", axis=1)

ring_area = np.pi * (ring_df['ring_end'] ** 2 - ring_df['ring_start'] ** 2)
ring_df['per_sq_mi'] = (ring_df['incidents'] / ring_area).round(0)

ring_df.pivot(index='ring_label', columns='period', values='incidents')

period,Day,Night
ring_label,,
0.00–0.05 mi,0,0
0.05–0.10 mi,1,0
0.10–0.15 mi,31,3
0.15–0.20 mi,16,21
0.20–0.25 mi,15,20
0.25–0.30 mi,26,21
0.30–0.35 mi,45,48
0.35–0.40 mi,46,39
0.40–0.45 mi,68,39


### 4. Chart

Hover over any ring to highlight it and see the day and night numbers side by side.

The red dashed line marks the approximate edge of main campus. Campus isn't a perfect circle around Snell, so this is an estimate: most of main campus (West Village, Ruggles, Columbus Ave) is within about 0.25 miles. Change `CAMPUS_EDGE_MI` to move it.

In [5]:
CAMPUS_EDGE_MI = 0.25   # approximate edge of main campus

period_colors = alt.Scale(domain=['Day', 'Night'], range=['#E69F00', '#1F4E79'])

# One row per ring with day and night side by side, for the tooltip
wide = ring_df.pivot(index=['ring_start', 'ring_end', 'ring_label'], columns='period',
                     values=['incidents', 'per_sq_mi', 'top_offense'])
wide.columns = [f'{period.lower()}_{col}' for col, period in wide.columns]
wide = wide.reset_index()

hover = alt.selection_point(fields=['ring_start'], on='pointerover',
                            clear='pointerout', empty=False)

X_TITLE = 'Distance from Snell Library (miles, 0.05-mile rings)'

x_axis = alt.X('ring_start:Q',
               title=X_TITLE,
               scale=alt.Scale(domain=[0, RADIUS_MI]),
               axis=alt.Axis(format='.2f', values=list(rings) + [RADIUS_MI]))

base = alt.Chart(ring_df).encode(
    x=x_axis,
    y=alt.Y('incidents:Q', title='Number of incidents'),
    color=alt.Color('period:N', scale=period_colors,
                    legend=alt.Legend(title='Time of day', orient='top-left')),
)

lines = base.mark_line(strokeWidth=2.5)

# Points appear only on the hovered ring
points = base.mark_point(filled=True, size=70).encode(
    opacity=alt.condition(hover, alt.value(1), alt.value(0)),
)

# Red dashed line at the edge of campus
edge_df = pd.DataFrame({'edge': [CAMPUS_EDGE_MI], 'label': ['Approx. campus edge']})
campus_edge = alt.Chart(edge_df).mark_rule(color='#D62728', strokeDash=[6, 4], strokeWidth=2).encode(
    x=alt.X('edge:Q', title=X_TITLE))
campus_label = alt.Chart(edge_df).mark_text(
    color='#D62728', align='left', dx=5, dy=8, fontWeight='bold').encode(
    x=alt.X('edge:Q', title=X_TITLE), y=alt.value(0), text='label:N')

# Invisible full-height band for each ring: catches the hover and shows the tooltip
bands = alt.Chart(wide).mark_rect(color='gray').encode(
    x=x_axis,
    x2='ring_end:Q',
    opacity=alt.condition(hover, alt.value(0.12), alt.value(0.001)),
    tooltip=[
        alt.Tooltip('ring_label:N', title='Ring'),
        alt.Tooltip('day_incidents:Q', title='Day incidents'),
        alt.Tooltip('night_incidents:Q', title='Night incidents'),
        alt.Tooltip('day_per_sq_mi:Q', title='Day per sq mi', format=',.0f'),
        alt.Tooltip('night_per_sq_mi:Q', title='Night per sq mi', format=',.0f'),
        alt.Tooltip('day_top_offense:N', title='Most common (day)'),
        alt.Tooltip('night_top_offense:N', title='Most common (night)'),
    ],
).add_params(hover)

viz5 = (campus_edge + campus_label + lines + points + bands).properties(
    width=650,
    height=380,
    title=alt.Title(
        'Crime near Northeastern by distance from Snell Library',
        subtitle='Serious (UCR Part One) incidents within 0.5 miles, day vs. night. Hover over a ring for details.',
    ),
)

viz5

alt.LayerChart(...)

In [6]:
# Saved as JSON for the website (walk-home.html loads charts/distance.json)
viz5.save('../charts/distance.json')

## Paragraph outlines

### Takeaway paragraph (webpage)
1. **Question:** does crime go up as you walk away from campus?
2. **Answer in one line:** yes, but more gradually than the raw counts suggest once you account for area. Per square mile, crime is about 35–40% denser outside the campus line than inside it.
3. **Evidence:**
   - Almost nothing within 0.1 mi of Snell.
   - Daytime spike at 0.10–0.15 mi: 31 day vs. 3 night. 30 of the 31 are on Huntington Ave, and 29 are larceny.
   - Inside the campus line (0.25 mi), per square mile: about 321 day and 224 night. Outside it: about 435 day and 319 night.
4. **What it means for students:** what to watch for walking home at night once you leave the campus core.
5. **Caveat:** these are counts, not rates. On-campus incidents handled by NUPD aren't in the data.

### Design paragraph (Word doc)
- **Chart type:** why a line chart works for an ordered distance variable, and why rings are 0.05 mi wide.
- **Encodings:** x = distance from Snell, y = incident count, color = day or night.
- **Color choices:** amber for day and dark blue for night as an intuitive pairing, red dashed line for the campus edge.
- **Interaction:** the hover band, and why one tooltip shows day and night together. Mention the incidents-per-sq-mi field and why you added it (outer rings cover more area).
- **Data decisions:** 0.5 mi radius, 6 AM and 6 PM as the day/night cutoffs, the campus edge as an estimate, and empty rings plotted as 0.

## Takeaway paragraph (webpage)

This visualization helps answer the question "does crime increase as you walk away from the NEU Boston campus?" The answer to this question is yes, but more gradually than the raw counts make it seem once you account for area sizes, which users can see by hovering over each ring to view the incidents per square mile. According to the chart, there is almost no crime within 0.1 miles of Snell Library, the approximate center of the Boston campus, which is likely because on-campus incidents are typically handled by NUPD and are not included in this Boston Police Department data. Moving past 0.1 miles, there is a jump to 31 incidents during the day but only 3 at night for 0.1-0.15 miles from Snell. Of these daytime incidents, almost all of them occurred on Huntington Ave (30 of the 31) and most were larceny (29 instances). At night, crime stays low right at the edge of campus and then starts to pick up from about 0.15 miles outward. Further, there is a line drawn for the approximate edge of campus, about 0.25 miles from Snell, which helps visualize on vs off campus crime. Using the data, we calculated that inside campus there were about 321 incidents per square mile during the day and 224 at night, while outside campus there were about 435 during the day and 319 at night, so crime is around 35-40% denser once you leave campus. However, we must note that this data is based on counts, not rates. Because there is no foot traffic data, busy areas like Huntington Ave may look more dangerous just because more people are there. All in all, the key takeaway here is that crime near campus at night is low, but students should be more aware once they walk past the edge of campus, since that is where nighttime crime starts to rise.

## Design paragraph (Word doc)

For this visualization, we chose a line chart because distance from Snell is an ordered variable, so a line makes it easy to see how crime changes as you walk farther from campus. We grouped incidents into 0.05-mile rings, which are small enough to show changes but large enough that each ring still has enough incidents to compare. The x-axis shows distance from Snell, the y-axis shows the number of incidents, and color separates day and night, with amber for day and dark blue for night since those colors are easy to tell apart. Further, we added a red dashed line at about 0.25 miles to mark the approximate edge of campus so users can compare on vs off campus crime. When the user hovers over a ring, it highlights and a tooltip shows the day and night numbers together, along with incidents per square mile, since outer rings cover more area and would otherwise look worse than they are. All in all, we kept the same 0.5-mile radius as our other visualizations, used 6 AM and 6 PM as the day and night cutoffs, and plotted empty rings as 0 so the lines do not skip any distances.